# Model Comparison: Mule Account Detection

This notebook trains and compares 6 models for mule account detection:
1. Logistic Regression
2. Random Forest
3. XGBoost
4. CatBoost
5. LightGBM
6. Neural Network

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
from sklearn.metrics import confusion_matrix, f1_score, brier_score_loss
from sklearn.calibration import calibration_curve
from scipy.stats import wilcoxon
import joblib
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print('Imports OK')

## 1. Setup & Data Loading

In [ ]:
# Load feature matrix and labels
features_path = Path('../outputs/features_matrix.parquet')
labels_path = Path('../data/train_labels.csv')

if features_path.exists():
    X_df = pd.read_parquet(features_path)
    print(f'Features loaded: {X_df.shape}')
else:
    print(f'Features file not found at {features_path}, generating synthetic data for demo')
    np.random.seed(42)
    n_samples = 5000
    n_features = 57
    X_df = pd.DataFrame(
        np.random.randn(n_samples, n_features),
        columns=[f'feature_{i}' for i in range(n_features)]
    )

if labels_path.exists():
    labels_df = pd.read_csv(labels_path)
    y = labels_df['label'].values if 'label' in labels_df.columns else labels_df.iloc[:, -1].values
    # Align indices
    if 'account_id' in labels_df.columns and 'account_id' in X_df.columns:
        merged = X_df.merge(labels_df[['account_id', 'label']], on='account_id')
        y = merged['label'].values
        X_df = merged.drop(columns=['account_id', 'label'])
    else:
        X_df = X_df.iloc[:len(y)]
        y = y[:len(X_df)]
    print(f'Labels loaded: {len(y)} (positive rate: {y.mean():.3f})')
else:
    print('Labels file not found, generating synthetic labels')
    y = np.random.binomial(1, 0.1, len(X_df))
    print(f'Synthetic labels: {len(y)} (positive rate: {y.mean():.3f})')

# Drop non-numeric columns
feature_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()
if 'account_id' in feature_cols:
    feature_cols.remove('account_id')
X = X_df[feature_cols].values
feature_names = feature_cols
print(f'Final feature matrix: {X.shape}')

## 2. Train/Validation Split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]} samples (positive rate: {y_train.mean():.3f})')
print(f'Val:   {X_val.shape[0]} samples (positive rate: {y_val.mean():.3f})')

## 3. Train All 6 Models

In [ ]:
from src.models.trainer import ModelTrainer

trainer = ModelTrainer()

# Train all models with 20 Optuna trials each (notebook speed)
print('Training all 6 models with 20 Optuna trials each...')
print('This may take several minutes.\n')

all_results = trainer.train_all_models(
    X_train, y_train, X_val, y_val, n_trials=20
)

print(f'\nTrained {len(all_results)} models successfully')
for name, res in all_results.items():
    print(f'  {name}: AUC-ROC = {res["score"]:.4f}')

## 4. Model Comparison Table

In [ ]:
from src.models.evaluator import ModelEvaluator

evaluator = ModelEvaluator()

comparison_data = []
model_probs = {}

for name, res in all_results.items():
    wrapper = res['model']
    y_prob = wrapper.predict_proba(X_val)
    model_probs[name] = y_prob
    
    metrics = evaluator.evaluate(y_val, y_prob)
    metrics['model'] = name
    comparison_data.append(metrics)

comparison_df = pd.DataFrame(comparison_data).set_index('model')
comparison_df = comparison_df.sort_values('auc_roc', ascending=False)

display_cols = ['auc_roc', 'auc_pr', 'f1', 'precision', 'recall', 'brier_score']
available_cols = [c for c in display_cols if c in comparison_df.columns]
print('Model Comparison Table:')
comparison_df[available_cols].round(4)

## 5. Overlaid ROC Curves

In [ ]:
colors = px.colors.qualitative.Set2

fig = go.Figure()
for i, (name, y_prob) in enumerate(model_probs.items()):
    fpr, tpr, _ = roc_curve(y_val, y_prob)
    auc = roc_auc_score(y_val, y_prob)
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr,
        name=f'{name} (AUC={auc:.4f})',
        line=dict(color=colors[i % len(colors)], width=2)
    ))

fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    name='Random', line=dict(dash='dash', color='gray', width=1)
))

fig.update_layout(
    title='ROC Curves - All Models',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    width=800, height=600,
    legend=dict(x=0.5, y=0.05)
)
fig.show()

## 6. Overlaid Precision-Recall Curves

In [ ]:
fig = go.Figure()
for i, (name, y_prob) in enumerate(model_probs.items()):
    precision, recall, _ = precision_recall_curve(y_val, y_prob)
    ap = average_precision_score(y_val, y_prob)
    fig.add_trace(go.Scatter(
        x=recall, y=precision,
        name=f'{name} (AP={ap:.4f})',
        line=dict(color=colors[i % len(colors)], width=2)
    ))

# Baseline
baseline = y_val.mean()
fig.add_trace(go.Scatter(
    x=[0, 1], y=[baseline, baseline],
    name=f'Baseline ({baseline:.3f})', line=dict(dash='dash', color='gray', width=1)
))

fig.update_layout(
    title='Precision-Recall Curves - All Models',
    xaxis_title='Recall',
    yaxis_title='Precision',
    width=800, height=600,
    legend=dict(x=0.05, y=0.05)
)
fig.show()

## 7. Confusion Matrices (2x3 Grid)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, (name, y_prob) in enumerate(model_probs.items()):
    if i >= 6:
        break
    # Use optimal threshold from PR curve
    prec, rec, thresholds = precision_recall_curve(y_val, y_prob)
    f1_scores = 2 * prec * rec / (prec + rec + 1e-8)
    best_idx = np.argmax(f1_scores[:-1])
    best_threshold = thresholds[best_idx]
    y_pred = (y_prob >= best_threshold).astype(int)
    
    cm = confusion_matrix(y_val, y_pred)
    im = axes[i].imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    axes[i].set_title(f'{name}\n(threshold={best_threshold:.3f})', fontsize=11)
    
    # Annotate
    for row in range(cm.shape[0]):
        for col in range(cm.shape[1]):
            axes[i].text(col, row, format(cm[row, col], 'd'),
                        ha='center', va='center',
                        color='white' if cm[row, col] > cm.max()/2 else 'black')
    
    axes[i].set_ylabel('True')
    axes[i].set_xlabel('Predicted')
    axes[i].set_xticks([0, 1])
    axes[i].set_yticks([0, 1])
    axes[i].set_xticklabels(['Legit', 'Mule'])
    axes[i].set_yticklabels(['Legit', 'Mule'])

plt.suptitle('Confusion Matrices - All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
Path('../outputs/plots').mkdir(parents=True, exist_ok=True)
plt.savefig('../outputs/plots/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Calibration Plots (Top 3 Models)

In [ ]:
# Get top 3 models by AUC-ROC
top3 = comparison_df.index[:3].tolist()

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.plot([0, 1], [0, 1], 'k--', label='Perfectly Calibrated')

for name in top3:
    y_prob = model_probs[name]
    prob_true, prob_pred = calibration_curve(y_val, y_prob, n_bins=10, strategy='uniform')
    brier = brier_score_loss(y_val, y_prob)
    ax.plot(prob_pred, prob_true, 's-', label=f'{name} (Brier={brier:.4f})')

ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Plots - Top 3 Models')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/plots/calibration_top3.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Statistical Significance (5-Fold CV + Wilcoxon)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from scipy.stats import wilcoxon

# 5-fold CV for top 3 models
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = {name: [] for name in top3}

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_fold_train, X_fold_val = X[train_idx], X[val_idx]
    y_fold_train, y_fold_val = y[train_idx], y[val_idx]
    
    for name in top3:
        wrapper = all_results[name]['model']
        params = all_results[name]['params']
        
        # Rebuild and retrain with best params
        wrapper.build_model(params)
        wrapper.fit(X_fold_train, y_fold_train, X_fold_val, y_fold_val)
        y_prob_fold = wrapper.predict_proba(X_fold_val)
        score = roc_auc_score(y_fold_val, y_prob_fold)
        cv_scores[name].append(score)
    
    print(f'Fold {fold+1}/5 complete')

# Display CV scores
cv_df = pd.DataFrame(cv_scores)
print('\n5-Fold CV AUC-ROC Scores:')
print(cv_df.round(4))
print(f'\nMeans: {cv_df.mean().round(4).to_dict()}')
print(f'Stds:  {cv_df.std().round(4).to_dict()}')

# Pairwise Wilcoxon tests
print('\nPairwise Wilcoxon Signed-Rank Tests:')
for i in range(len(top3)):
    for j in range(i+1, len(top3)):
        try:
            stat, p_val = wilcoxon(cv_scores[top3[i]], cv_scores[top3[j]])
            sig = '***' if p_val < 0.01 else '**' if p_val < 0.05 else '*' if p_val < 0.1 else 'ns'
            print(f'  {top3[i]} vs {top3[j]}: p={p_val:.4f} ({sig})')
        except Exception as e:
            print(f'  {top3[i]} vs {top3[j]}: Could not compute ({e})')

## 10. Feature Importance (Best Model - Top 20)

In [ ]:
best_model_name = comparison_df.index[0]
best_wrapper = all_results[best_model_name]['model']

# Retrain best model on full train set with best params
best_wrapper.build_model(all_results[best_model_name]['params'])
best_wrapper.fit(X_train, y_train, X_val, y_val)

importance = best_wrapper.get_feature_importance()

if importance:
    imp_df = pd.DataFrame([
        {'feature': feature_names[int(k)] if isinstance(k, (int, np.integer)) and int(k) < len(feature_names) else str(k),
         'importance': v}
        for k, v in importance.items()
    ]).sort_values('importance', ascending=False).head(20)
    
    fig = go.Figure(go.Bar(
        x=imp_df['importance'].values[::-1],
        y=imp_df['feature'].values[::-1],
        orientation='h'
    ))
    fig.update_layout(
        title=f'Top 20 Feature Importances ({best_model_name})',
        xaxis_title='Importance',
        yaxis_title='Feature',
        width=800, height=600
    )
    fig.show()
else:
    print(f'Feature importance not available for {best_model_name}')

## 11. Learning Curves (Best Model)

In [ ]:
fractions = [0.1, 0.25, 0.5, 0.75, 1.0]
train_scores = []
val_scores = []

for frac in fractions:
    n = int(len(X_train) * frac)
    X_sub = X_train[:n]
    y_sub = y_train[:n]
    
    best_wrapper.build_model(all_results[best_model_name]['params'])
    best_wrapper.fit(X_sub, y_sub, X_val, y_val)
    
    train_prob = best_wrapper.predict_proba(X_sub)
    val_prob = best_wrapper.predict_proba(X_val)
    
    train_auc = roc_auc_score(y_sub, train_prob)
    val_auc = roc_auc_score(y_val, val_prob)
    
    train_scores.append(train_auc)
    val_scores.append(val_auc)
    print(f'{frac*100:.0f}% data ({n} samples): Train AUC={train_auc:.4f}, Val AUC={val_auc:.4f}')

fig = go.Figure()
fig.add_trace(go.Scatter(x=[f'{f*100:.0f}%' for f in fractions], y=train_scores, name='Train AUC', mode='lines+markers'))
fig.add_trace(go.Scatter(x=[f'{f*100:.0f}%' for f in fractions], y=val_scores, name='Validation AUC', mode='lines+markers'))
fig.update_layout(
    title=f'Learning Curves ({best_model_name})',
    xaxis_title='Training Data Fraction',
    yaxis_title='AUC-ROC',
    width=800, height=500
)
fig.show()

## 12. Save Best Model

In [ ]:
model_dir = Path('../outputs/models')
model_dir.mkdir(parents=True, exist_ok=True)

# Retrain best model on full train set
best_wrapper.build_model(all_results[best_model_name]['params'])
best_wrapper.fit(X_train, y_train, X_val, y_val)

save_path = model_dir / 'best_model.joblib'
best_wrapper.save(save_path)

# Save metadata
import json
metadata = {
    'model_name': best_model_name,
    'params': {k: str(v) if not isinstance(v, (int, float, str, bool, list)) else v 
               for k, v in all_results[best_model_name]['params'].items()},
    'val_auc_roc': float(roc_auc_score(y_val, best_wrapper.predict_proba(X_val))),
    'feature_names': feature_names,
    'n_features': len(feature_names),
    'n_train_samples': len(X_train),
}

with open(model_dir / 'best_model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Best model saved: {best_model_name}')
print(f'Path: {save_path}')
print(f'Val AUC-ROC: {metadata["val_auc_roc"]:.4f}')